In [1]:
import os

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "org.apache.hadoop:hadoop-aws:3.3.4,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
    "org.postgresql:postgresql:42.7.3 "
    "pyspark-shell"
)


from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("outputevents-session") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.DefaultAWSCredentialsProviderChain") \
    .getOrCreate()




In [2]:
df = spark.read.text("s3a://kafka-consumer-spark/consumer/outputevents/")
df.show()

+--------------------+
|               value|
+--------------------+
|[{"row_id": 29042...|
|[{"row_id": 29044...|
|[{"row_id": 31513...|
|[{"row_id": 31510...|
|[{"row_id": 36601...|
|[{"row_id": 40210...|
|[{"row_id": 31515...|
|[{"row_id": 30059...|
|[{"row_id": 14768...|
|[{"row_id": 28901...|
|[{"row_id": 28378...|
|[{"row_id": 28379...|
|[{"row_id": 34223...|
|[{"row_id": 31508...|
|[{"row_id": 33112...|
|[{"row_id": 28377...|
|[{"row_id": 34224...|
|[{"row_id": 31337...|
|[{"row_id": 29060...|
|[{"row_id": 30283...|
+--------------------+
only showing top 20 rows



In [4]:
df_test = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://3.94.79.85:5432/mimic") \
    .option("dbtable", "outputevents") \
    .option("user", "postgres") \
    .option("password", "123456") \
    .option("driver", "org.postgresql.Driver") \
    .load()

df_test.show(5)

+------+----------+-------+----------+---------+------+-----+--------+---------+----+-------+---------+-------+----------+
|row_id|subject_id|hadm_id|icustay_id|charttime|itemid|value|valueuom|storetime|cgid|stopped|newbottle|iserror|item_label|
+------+----------+-------+----------+---------+------+-----+--------+---------+----+-------+---------+-------+----------+
+------+----------+-------+----------+---------+------+-----+--------+---------+----+-------+---------+-------+----------+



In [5]:
d_items = spark.read \
    .option("header", True) \
    .csv("s3a://aws-glue-trying/SourceData/mimic_Demo/D_ITEMS.csv")

In [6]:
from pyspark.sql.functions import col

d_items = d_items.withColumn(
    "itemid",
    col("itemid").cast("int")
)

In [9]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

schema = StructType([
    StructField("cgid", LongType(), True),
    StructField("charttime", StringType(), True),
    StructField("hadm_id", LongType(), True),
    StructField("icustay_id", LongType(), True),
    StructField("iserror", StringType(), True),
    StructField("itemid", LongType(), True),
    StructField("newbottle", StringType(), True),
    StructField("row_id", LongType(), True),
    StructField("stopped", StringType(), True),
    StructField("storetime", StringType(), True),
    StructField("subject_id", LongType(), True),
    StructField("value", DoubleType(), True),
    StructField("valueuom", StringType(), True)
])

In [10]:
df_one = spark.readStream \
    .schema(schema) \
    .format("json") \
    .option("maxFilesPerTrigger", 1) \
    .load("s3a://kafka-consumer-spark/consumer/outputevents/")

In [16]:
from pyspark.sql.functions import col, coalesce, lit, to_timestamp
def process_batch(df, batch_id):
    
    df_clean = df.drop("stopped", "newbottle", "iserror")
    
    df_clean = df_clean \
        .withColumn("charttime", to_timestamp(col("charttime"))) \
        .withColumn("storetime", to_timestamp(col("storetime")))
    
    
    df_clean = df_clean.select(
        col("row_id").cast("int").alias("row_id"),
        col("subject_id").cast("int").alias("subject_id"),
        col("hadm_id").cast("int").alias("hadm_id"),
        col("icustay_id").cast("int").alias("icustay_id"),
        col("charttime").alias("charttime"),
        col("itemid").cast("int").alias("itemid"),
        col("value").cast("double").alias("value"),
        col("valueuom").cast("string").alias("valueuom"),
        col("storetime").alias("storetime"),
        col("cgid").cast("long").alias("cgid"),
    )
    
    
    df_joined = (
        df_clean.join(
            d_items.select(
                col("itemid"),
                col("label").alias("item_label")
            ),
            on="itemid",
            how="left"
        )
        .withColumn(
            "item_label",
            coalesce(col("item_label"), lit("Item"))
        )
    )
    
    df_joined.write \
        .format("jdbc") \
        .option("url", "jdbc:postgresql://3.94.79.85:5432/mimic") \
        .option("dbtable", "outputevents") \
        .option("user", "postgres") \
        .option("password", "123456") \
        .option("driver", "org.postgresql.Driver") \
        .mode("append") \
        .save()

In [17]:
query = df_one.writeStream \
    .trigger(processingTime="5 seconds") \
    .foreachBatch(process_batch) \
    .option("checkpointLocation", "checkpointoutput") \
    .start()

query.awaitTermination()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [18]:
query.lastProgress["sources"]

[{'description': 'FileStreamSource[s3a://kafka-consumer-spark/consumer/outputevents]',
  'startOffset': {'logOffset': 112},
  'endOffset': {'logOffset': 112},
  'latestOffset': None,
  'numInputRows': 0,
  'inputRowsPerSecond': 0.0,
  'processedRowsPerSecond': 0.0}]

In [19]:
spark.stop()